Two sources:
1. `multilingualmc/dictionary_collection/mturk/analysis`
2. `multilingualmc/data_term_gold/processed`

Manually change the column name from "word" to "English", and from "validated_translation" to "Chinese".

In [33]:
import pandas as pd
import json
data_gold = pd.read_json("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/data_term_gold/processed/chinese.json").rename(columns={"term_reformatted": "word", "term_Chinese": "validated_translation"}, inplace=False).drop_duplicates(subset='word', inplace=False)
data_gold['word_lower'] = data_gold['word'].str.lower()
data_6060 = pd.read_csv("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dictionary_collection/mturk/analysis/annotation_results_6060/Chinese_validated.csv").rename(columns={"validated_translation": "validated_translation_old", "gold": "validated_translation"}, inplace=False).drop_duplicates(subset='word', inplace=False)
data_mturk = pd.read_csv("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dictionary_collection/mturk/analysis/annotation_results_crawled/Chinese_validated.csv").drop_duplicates(subset='word', inplace=False)

In [34]:
concat_df = pd.concat([data_6060[['word', 'validated_translation']], data_mturk[['word', 'validated_translation']]], axis=0, ignore_index=True)
concat_df['word_lower'] = concat_df['word'].str.lower()
concat_df

,word,validated_translation,word_lower
0,AI,人工智能,ai
1,API,API,api
2,AQA,AQA,aqa
3,ARENA,ARENA,arena
4,Additionally,此外,additionally
...,...,...,...
4985,zero-shot prompting,零样本提示,zero-shot prompting
4986,zero-shot reasoning,零样本推理,zero-shot reasoning
4987,zero-shot setting,零样本设置,zero-shot setting
4988,zero-shot transfer,零样本迁移,zero-shot transfer


In [35]:
lang = 'Chinese'
concat_df_ref = pd.read_csv(f"/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dictionary_collection/mturk/analysis/annotation_results_crawled/{lang}_validated_>0.5.csv")[['word', 'validated_translation']]

concat_df_ref = concat_df_ref.dropna(subset=['validated_translation'])

translation_dict = concat_df_ref.set_index("word")["validated_translation"].to_dict()

# Replace "validated_translation" in df where the "word" exists in df_ref
concat_df["validated_translation"] = concat_df.apply(
    lambda row: translation_dict[row["word"]] if row["word"] in translation_dict else row["validated_translation"],
    axis=1
)

In [36]:
df_itst = pd.merge(data_gold[['word', 'word_lower', 'validated_translation']], concat_df[['word', 'word_lower', 'validated_translation']], on='word_lower')
df_itst[df_itst['validated_translation_x'] == df_itst['validated_translation_y']].shape[0] / df_itst.shape[0], df_itst[df_itst['validated_translation_x'] == df_itst['validated_translation_y']].shape[0], df_itst.shape[0]
df_itst

,word_x,word_lower,validated_translation_x,word_y,validated_translation_y
0,accuracy,accuracy,准确率,Accuracy,准确率
1,acquisition function,acquisition function,采集函数,acquisition function,获取函数
2,activation,activation,活性值,activation,激活
3,activation function,activation function,激活函数,activation function,激活函数
4,active learning,active learning,主动学习,Active learning,主动学习
...,...,...,...,...,...
795,prediction,prediction,预测,prediction,预测
796,prediction accuracy,prediction accuracy,预测准确率,prediction accuracy,预测准确率
797,predictor,predictor,预测器,predictor,预测器
798,protein folding,protein folding,蛋白折叠,protein folding,蛋白质折叠


In [37]:
merged_df = pd.merge(concat_df[['word', 'word_lower', 'validated_translation']], data_gold[['word', 'word_lower', 'validated_translation']], on="word_lower", how='outer', suffixes=('_df2', '_df1'))
merged_df

,word_df2,word_lower,validated_translation_df2,word_df1,validated_translation_df1
0,NaN,0-1 loss function,NaN,0-1 loss function,0-1损失函数
1,10-fold cross validation,10-fold cross validation,10折交叉验证,NaN,NaN
2,1D convolution,1d convolution,一维卷积,NaN,NaN
3,2 norm,2 norm,2范数,NaN,NaN
4,2D convolution,2d convolution,二维卷积,NaN,NaN
...,...,...,...,...,...
6640,zero-shot transfer learning,zero-shot transfer learning,零样本迁移学习,NaN,NaN
6641,Zipf,zipf,齐普夫,NaN,NaN
6642,Zipf distribution,zipf distribution,齐普夫分布,NaN,NaN
6643,Zipf's law,zipf's law,齐普夫定律,Zipf's law,齐普夫定律


In [38]:
df_itst[df_itst['validated_translation_x'] != df_itst['validated_translation_y']]

,word_x,word_lower,validated_translation_x,word_y,validated_translation_y
1,acquisition function,acquisition function,采集函数,acquisition function,获取函数
2,activation,activation,活性值,activation,激活
5,actor,actor,演员,actor,行动者
6,actor-critic method,actor-critic method,演员-评论员法,actor-critic method,Actor-Critic 方法
7,adversarial,adversarial,对抗,adversarial,对抗性
...,...,...,...,...,...
773,eigenfunction,eigenfunction,特征函数,eigenfunction,本征函数
774,facial recognition,facial recognition,面部识别,facial recognition,人脸识别
790,Monte Carlo tree search,monte carlo tree search,蒙特卡洛树搜索,Monte Carlo Tree Search,蒙特卡罗树搜索
792,network architecture,network architecture,网络结构,network architecture,网络架构


In [39]:
merged_df['validated_translation'] = merged_df['validated_translation_df1'].combine_first(merged_df['validated_translation_df2'])
merged_df['word'] = merged_df['word_df1'].combine_first(merged_df['word_df2'])
merged_df

,word_df2,word_lower,validated_translation_df2,word_df1,validated_translation_df1,validated_translation,word
0,NaN,0-1 loss function,NaN,0-1 loss function,0-1损失函数,0-1损失函数,0-1 loss function
1,10-fold cross validation,10-fold cross validation,10折交叉验证,NaN,NaN,10折交叉验证,10-fold cross validation
2,1D convolution,1d convolution,一维卷积,NaN,NaN,一维卷积,1D convolution
3,2 norm,2 norm,2范数,NaN,NaN,2范数,2 norm
4,2D convolution,2d convolution,二维卷积,NaN,NaN,二维卷积,2D convolution
...,...,...,...,...,...,...,...
6640,zero-shot transfer learning,zero-shot transfer learning,零样本迁移学习,NaN,NaN,零样本迁移学习,zero-shot transfer learning
6641,Zipf,zipf,齐普夫,NaN,NaN,齐普夫,Zipf
6642,Zipf distribution,zipf distribution,齐普夫分布,NaN,NaN,齐普夫分布,Zipf distribution
6643,Zipf's law,zipf's law,齐普夫定律,Zipf's law,齐普夫定律,齐普夫定律,Zipf's law


In [40]:
merged_df[merged_df['word_lower'] == 'activation']

,word_df2,word_lower,validated_translation_df2,word_df1,validated_translation_df1,validated_translation,word
70,activation,activation,激活,activation,活性值,活性值,activation


In [41]:
merged_df = merged_df[['word', 'validated_translation']].drop_duplicates(subset='word', inplace=False).sort_values(by='word')
merged_df.to_csv("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dictionary_collection/mturk/analysis/annotation_final/Chinese.csv")